# Assignment 2: Classification and Regression

**Total Marks:** 10  
**Level:** Intermediate  
**Topics:** Training, hyperparameter tuning, and evaluation of classification and regression algorithms

---

## Instructions

This assignment contains two parts:
1. **Part A: Classification on Unbalanced Data (5 marks)**
2. **Part B: Regression Models (5 marks)**

**Important:**
- Datasets are pre-generated and provided for you
- Store results in the **exact variable names** specified in each task
- Execute all cells before submission
- Otter-Grader will automatically test your outputs

---

## Setup: Imports and Dataset Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, auc, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

---

# PART A: CLASSIFICATION ON UNBALANCED DATA (5 Marks)

---

## Classification - Data Preparation (PROVIDED)

In [ ]:
# Generate unbalanced classification dataset (90-10 class imbalance)
X_class, y_class = make_classification(n_samples=1000, n_features=20, n_informative=15,
                                       n_redundant=5, weights=[0.9, 0.1], random_state=42)

# Split data with stratification 80:20
X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(X_class, y_class, test_size=0.20, train_size=0.80, stratify=y_class, random_state=42)

# Standardize features
scaler_class = StandardScaler()
X_class_train_scaled = scaler_class.fit_transform(X_class_train)
X_class_test_scaled = scaler_class.fit_transform(X_class_test) 

print("Classification Dataset:")
print(f"Training samples: {X_class_train_scaled.shape[0]}, Testing samples: {X_class_test_scaled.shape[0]}")
print(f"Features: {X_class_train_scaled.shape[1]}")
print(f"Class distribution (train): {np.bincount(y_class_train)}")
print(f"Class imbalance ratio: {np.bincount(y_class)[0] / np.bincount(y_class)[1]:.2f}:1")

## Classification - Task 1: Baseline Models (1.5 Marks)

Train 5 classification models with default hyperparameters:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. Support Vector Machine (SVM)
5. Gaussian Naive Bayes

**Expected Output Variable: `classification_baseline_results`**
- Create a pandas DataFrame with:
  - Index: Model names
  - Columns: "Accuracy", "ROC-AUC"
  - Values: Metrics calculated on test set

In [ ]:
# BEGIN QUESTION
# name: classification_baseline_models
# points: 1.5

logistic_regression = LogisticRegression()
decision_classifier = DecisionTreeClassifier()
random_forest_classifier = RandomForestClassifier()
support_vector_machine = SVC(probability=True)
gaussian_naive_classifier = GaussianNB()

classification_models = {
    "logistic_regression": logistic_regression,
    "decision_classifier": decision_classifier,
    "random_forest_classifier": random_forest_classifier,
    "support_vector_machine": support_vector_machine,
    "gaussian_naive_classifier": gaussian_naive_classifier
}
data_dict = {"logistic_regression": {}, "decision_classifier": {}, 
             "random_forest_classifier": {}, "support_vector_machine": {},
             "gaussian_naive_classifier": {}}

for name, model in classification_models.items():
    model.fit(X_class_train_scaled, y_class_train)
    y_predict = logistic_regression.predict(X_class_test_scaled)
    data_dict[name]["Accuracy"]= accuracy_score(y_class_test, y_predict)
    data_dict[name]["ROC-AUC"] = roc_auc_score(y_class_test, y_predict)

# Train 5 baseline classification models
# Store results in DataFrame: classification_baseline_results
# Columns: "Accuracy", "ROC-AUC"
# Index: Model names

classification_baseline_results = pd.DataFrame(data_dict)
classification_baseline_results = classification_baseline_results.T
print("Classification Baseline Model Performance:")
print(classification_baseline_results)
# END QUESTION

In [ ]:
# BEGIN TESTS
assert classification_baseline_results is not None, "classification_baseline_results not created"
assert isinstance(classification_baseline_results, pd.DataFrame), "Must be a DataFrame"
assert classification_baseline_results.shape[0] == 5, "Must have 5 models"
assert classification_baseline_results.shape[1] == 2, "Must have 2 columns"
assert "Accuracy" in classification_baseline_results.columns
assert "ROC-AUC" in classification_baseline_results.columns
assert all(0 <= classification_baseline_results["Accuracy"]) & all(classification_baseline_results["Accuracy"] <=1)
assert all(0 <= classification_baseline_results["ROC-AUC"]) & all(classification_baseline_results["ROC-AUC"] <= 1)
# END TESTS

## Classification - Task 2: Hyperparameter Tuning (2.5 Marks)

Use GridSearchCV to tune hyperparameters for all 5 classification algorithms.

**Hyperparameter Grids:**
- **Logistic Regression:** `{'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}`
- **Decision Tree:** `{'max_depth': [None, 5, 10, 20], 'min_samples_split': [2, 5, 10], 'class_weight': [None, 'balanced']}`
- **Random Forest:** `{'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20], 'class_weight': [None, 'balanced']}`
- **SVM:** `{'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'class_weight': [None, 'balanced']}`
- **Naive Bayes:** `{'var_smoothing': [1e-9, 1e-8, 1e-7]}`

**Expected Output Variables:**
- `classification_tuned_models` (dict): Keys = model names, Values = best fitted models
- `classification_tuning_results` (dict): Keys = model names, Values = best hyperparameters

In [ ]:
# BEGIN QUESTION
# name: classification_hyperparameter_tuning
# points: 2.5

# Tune hyperparameters for all 5 classification algorithms
# Use GridSearchCV with cv=5, scoring='roc_auc'
# Store tuned models in: classification_tuned_models
# Store best params in: classification_tuning_results

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced']
}

param_grid_dt = {
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None, 'balanced']
}

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'class_weight': [None, 'balanced']
}

param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'class_weight': [None, 'balanced']
}

param_grid_nb = {
    'var_smoothing': [1e-9, 1e-8, 1e-7]
}

classification_tuned_models = {}  # Dict with tuned models
classification_tuning_results = {}  # Dict with best parameters

# Tune Logistic Regression
grid_search = GridSearchCV(
    estimator=logistic_regression,
    param_grid=param_grid_lr,
    cv=5,
    scoring='roc_auc'
)

grid_search.fit(X_class_train_scaled, y_class_train)

classification_tuned_models["logistic_regression"] = grid_search.best_estimator_
classification_tuning_results["logistic_regression"] = grid_search.best_params_

# Tune Decision Tree Classifier
grid_search = GridSearchCV(
    estimator=decision_classifier,
    param_grid=param_grid_dt,
    cv=5,
    scoring='roc_auc'
)

grid_search.fit(X_class_train_scaled, y_class_train)

classification_tuned_models["decision_tree_classifier"] = grid_search.best_estimator_
classification_tuning_results["decision_tree_classifier"] = grid_search.best_params_

# Tune Random Forest Classifier
grid_search = GridSearchCV(
    estimator=random_forest_classifier,
    param_grid=param_grid_rf,
    cv=5,
    scoring='roc_auc'
)

grid_search.fit(X_class_train_scaled, y_class_train)

classification_tuned_models["random_forest_classifier"] = grid_search.best_estimator_
classification_tuning_results["random_forest_classifier"] = grid_search.best_params_

# Tune SVM
grid_search = GridSearchCV(
    estimator=support_vector_machine,
    param_grid=param_grid_svm,
    cv=5,
    scoring='roc_auc'
)

grid_search.fit(X_class_train_scaled, y_class_train)

classification_tuned_models["support_vector_machine"] = grid_search.best_estimator_
classification_tuning_results["support_vector_machine"] = grid_search.best_params_

# Tune Gaussian Naive Bayes
grid_search = GridSearchCV(
    estimator=gaussian_naive_classifier,
    param_grid=param_grid_nb,
    cv=5,
    scoring='roc_auc'
)

grid_search.fit(X_class_train_scaled, y_class_train)

classification_tuned_models["gaussian_naive_classifier"] = grid_search.best_estimator_
classification_tuning_results["gaussian_naive_classifier"] = grid_search.best_params_

print("Best Hyperparameters Found:")
for model_name, params in classification_tuning_results.items():
    print(f"{model_name}: {params}")

# END QUESTION

In [ ]:
# BEGIN TESTS
assert isinstance(classification_tuned_models, dict), "Must be a dictionary"
assert isinstance(classification_tuning_results, dict), "Must be a dictionary"
assert len(classification_tuned_models) == 5, "Must have 5 models"
assert len(classification_tuning_results) == 5, "Must have 5 entries"
# END TESTS

## Classification - Task 3: Evaluation & Comparison (1.0 Mark)

Evaluate all tuned models and create visualizations.

**Expected Output Variables:**
- `classification_final_results` (DataFrame): Model names × ["Accuracy", "ROC-AUC"], sorted by ROC-AUC
- `classification_roc_plot` (figure): ROC curves for all 5 models

In [ ]:
# BEGIN QUESTION
# name: classification_evaluation_table
# points: 0.5

# Evaluate all tuned models on test set
# Create DataFrame: classification_final_results
# Sort by ROC-AUC in descending order

results = {}
for model_name, model in classification_tuned_models.items():
    y_predict = model.predict(X_class_test_scaled)
    if model_name not in results:
        results[model_name] = {}
    results[model_name]["Accuracy"] = accuracy_score(y_class_test, y_predict)
    results[model_name]["ROC-AUC"] = roc_auc_score(y_class_test, y_predict)

classification_final_results = pd.DataFrame(results).T
classification_final_results.sort_values(by="ROC-AUC", axis=0, ascending=False, inplace=True)

print("Classification Final Model Performance:")
print(classification_final_results)
print(f"\nBest Model: {classification_final_results.index[0]} with ROC-AUC = {classification_final_results.iloc[0]['ROC-AUC']:.4f}")
# END QUESTION

In [ ]:
# BEGIN TESTS
assert classification_final_results is not None
assert isinstance(classification_final_results, pd.DataFrame)
assert classification_final_results.shape[0] == 5
assert classification_final_results.shape[1] == 2
assert "Accuracy" in classification_final_results.columns
assert "ROC-AUC" in classification_final_results.columns
# END TESTS

In [ ]:
# BEGIN QUESTION
# name: classification_roc_curves
# points: 0.5

# Plot ROC curves for all 5 tuned models
# Store figure in: classification_roc_plot

classification_roc_plot = plt.figure(figsize=(10, 8))
colors = ['blue', 'green', 'red', 'purple', 'orange']

# Your code here to plot ROC curves
# For each model:
#   - Get probability predictions
#   - Calculate FPR, TPR using roc_curve()
#   - Calculate AUC and plot
results = {}
for model_name, model in classification_tuned_models.items():
    if model_name not in results:
        results[model_name] = {}
    results[model_name]["probs"] = model.predict_proba(X_class_test_scaled)
    y_predict = model.predict(X_class_test_scaled)
    results[model_name]["fpr_tpr"] = roc_curve(y_class_test, y_predict)
    results[model_name]["auc_score"] = roc_auc_score(y_class_test, y_predict)

# Plot random classifier baseline
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Classifier')

for index, (model_name, model) in enumerate(classification_tuned_models.items()):
    plt.plot(results[model_name]["fpr_tpr"][0], results[model_name]["fpr_tpr"][1], color=colors[index], lw=2, linestyle=':', label=model_name)


plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Classification Models', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()

# END QUESTION

In [ ]:
# BEGIN TESTS
assert classification_roc_plot is not None
# END TESTS

---

# PART B: REGRESSION MODELS (5 Marks)

---

## Regression - Data Preparation (PROVIDED)

In [ ]:
# Generate regression dataset
X_reg, y_reg = make_regression(n_samples=500, n_features=10, n_informative=8,
                               noise=20, random_state=42)

# Split data
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.20, train_size=0.80, random_state=42)

# Standardize features
scaler_reg = StandardScaler()
X_reg_train_scaled = scaler_reg.fit_transform(X_reg_train)
X_reg_test_scaled = scaler_reg.transform(X_reg_test)

print("Regression Dataset:")
print(f"Training samples: {X_reg_train_scaled.shape[0]}, Testing samples: {X_reg_test_scaled.shape[0]}")
print(f"Features: {X_reg_train_scaled.shape[1]}")
print(f"Target range: [{y_reg.min():.2f}, {y_reg.max():.2f}]")

## Regression - Task 1: Baseline Models (1.5 Marks)

Train 5 regression models with default hyperparameters:
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Polynomial Regression (degree=2)
5. Logarithmic Regression

**Expected Output Variable: `regression_baseline_results`**
- Create a pandas DataFrame with:
  - Index: Model names
  - Columns: "MSE", "R² Score"
  - Values: Metrics calculated on test set

In [ ]:
# BEGIN QUESTION
# name: regression_baseline_models
# points: 1.5
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Train 5 baseline regression models
# Store results in DataFrame: regression_baseline_results
# Columns: "MSE", "R² Score"
# Index: Model names
lr = LinearRegression()
rr = Ridge()
lasso = Lasso()
pr = PolynomialFeatures(degree=2)
logr = np.log1p(X_reg)

def safe_log1p(X):
    return np.log1p(np.clip(X, a_min=0, a_max=None))

reg_models = {
    "LinearRegression": Pipeline([
        ("lr", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("ridge", Ridge())
    ]),
    "Lasso": Pipeline([
        ("lasso", Lasso())
    ]),
    "PolynomialRegression": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("lr", LinearRegression())
    ]),
    "LogarithmicRegression": Pipeline([
        ("log", FunctionTransformer(safe_log1p, validate=True)),
        ("ridge", Ridge())
    ])
}

results = {}
for name, pipe in reg_models.items():
    pipe.fit(X_reg_train_scaled, y_reg_train)
    y_pred = pipe.predict(X_reg_test)
    results[name] = {
        "MSE": mean_squared_error(y_reg_test, y_pred),
        "R² Score": r2_score(y_reg_test, y_pred)
    }

# Hints:
# - Linear Regression: LinearRegression()
# - Ridge: Ridge()
# - Lasso: Lasso()
# - Polynomial: Use PolynomialFeatures(degree=2) then LinearRegression()
# - Logarithmic: Use np.log1p() transformation on features
# - Metrics: mean_squared_error(), r2_score()

regression_baseline_results = pd.DataFrame(results)
regression_baseline_results = regression_baseline_results.T
print("Regression Baseline Model Performance:")
print(regression_baseline_results)

# END QUESTION

In [ ]:
# BEGIN TESTS
assert regression_baseline_results is not None
assert isinstance(regression_baseline_results, pd.DataFrame)
assert regression_baseline_results.shape[0] == 5, "Must have 5 models"
assert regression_baseline_results.shape[1] == 2, "Must have 2 columns"
assert "MSE" in regression_baseline_results.columns
assert "R² Score" in regression_baseline_results.columns
# END TESTS

## Regression - Task 2: Hyperparameter Tuning (2.5 Marks)

Use GridSearchCV to tune hyperparameters for all 5 regression algorithms.

**Hyperparameter Grids:**
- **Linear Regression:** `{'fit_intercept': [True, False]}`
- **Ridge:** `{'alpha': [0.001, 0.01, 0.1, 1, 10]}`
- **Lasso:** `{'alpha': [0.001, 0.01, 0.1, 1, 10]}`
- **Polynomial:** `{'polynomialfeatures__degree': [1, 2, 3, 4]}`
- **Logarithmic:** `{'alpha': [0.001, 0.01, 0.1, 1]}`

**Expected Output Variables:**
- `regression_tuned_models` (dict): Keys = model names, Values = best fitted models
- `regression_tuning_results` (dict): Keys = model names, Values = best hyperparameters

In [ ]:
# BEGIN QUESTION
# name: regression_hyperparameter_tuning
# points: 2.5
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# Tune hyperparameters for all 5 regression algorithms
# Use GridSearchCV with cv=5, scoring='r2'
# Store tuned models in: regression_tuned_models
# Store best params in: regression_tuning_results

regression_tuned_models = {}  # Dict with tuned models
regression_tuning_results = {}  # Dict with best parameters

reg_models = {
    "LinearRegression": Pipeline([
        ("lr", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("ridge", Ridge())
    ]),
    "Lasso": Pipeline([
        ("lasso", Lasso())
    ]),
    "PolynomialRegression": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("lr", LinearRegression())
    ]),
    "LogarithmicRegression": Pipeline([
        ("log", FunctionTransformer(safe_log1p, validate=True)),
        ("ridge", Ridge())
    ])
}

# Tune Linear Regression
gv_lr = GridSearchCV(estimator=reg_models["LinearRegression"], param_grid={'lr__fit_intercept':[True, False]}, cv=5, scoring='r2')
gv_lr.fit(X_reg_train_scaled, y_reg_train)
regression_tuned_models["LinearRegression"] = gv_lr.best_estimator_
regression_tuning_results["LinearRegression"] = gv_lr.best_params_

# Tune Ridge Regression
gv_ridge = GridSearchCV(estimator=reg_models["Ridge"], param_grid={'ridge__alpha': [0.001, 0.01, 0.1, 1, 10]}, cv=5, scoring='r2')
gv_ridge.fit(X_reg_train_scaled, y_reg_train)
regression_tuned_models["Ridge"] = gv_ridge.best_estimator_
regression_tuning_results["Ridge"] = gv_ridge.best_params_

# Tune Lasso Regression
gv_lasso = GridSearchCV(estimator=reg_models["Lasso"], param_grid={'lasso__alpha': [0.001, 0.01, 0.1, 1, 10]}, cv=5, scoring='r2')
gv_lasso.fit(X_reg_train_scaled, y_reg_train)
regression_tuned_models["Lasso"] = gv_lasso.best_estimator_
regression_tuning_results["Lasso"] = gv_lasso.best_params_

# Tune Polynomial Regression
gv_plr = GridSearchCV(estimator=reg_models["PolynomialRegression"], param_grid={'poly__degree': [1, 2, 3, 4]}, cv=5, scoring='r2')
gv_plr.fit(X_reg_train_scaled, y_reg_train)
regression_tuned_models["PolynomialRegression"] = gv_plr.best_estimator_
regression_tuning_results["PolynomialRegression"] = gv_plr.best_params_

# Tune Logarithmic Regression
gv_log_reg = GridSearchCV(estimator=reg_models["LogarithmicRegression"], param_grid={'ridge__alpha': [0.001, 0.01, 0.1, 1]}, cv=5, scoring='r2')
gv_log_reg.fit(X_reg_train_scaled, y_reg_train)
regression_tuned_models["LogarithmicRegression"] = gv_log_reg.best_estimator_
regression_tuning_results["LogarithmicRegression"] = gv_log_reg.best_params_

print("Best Hyperparameters Found:")
for model_name, params in regression_tuning_results.items():
    print(f"{model_name}: {params}")

# END QUESTION

In [ ]:
# BEGIN TESTS
assert isinstance(regression_tuned_models, dict)
assert isinstance(regression_tuning_results, dict)
assert len(regression_tuned_models) == 5, "Must have 5 models"
assert len(regression_tuning_results) == 5, "Must have 5 entries"
# END TESTS

## Regression - Task 3: Evaluation & Comparison (1.0 Mark)

Evaluate all tuned models and create visualizations.

**Expected Output Variables:**
- `regression_final_results` (DataFrame): Model names × ["MSE", "R² Score"], sorted by R² Score
- `regression_comparison_plot` (figure): Comparison plot of model performance

In [ ]:
# BEGIN QUESTION
# name: regression_evaluation_table
# points: 0.5

# Evaluate all tuned models on test set
# Create DataFrame: regression_final_results
# Sort by R² Score in descending order
results = {}
for name, pipe in regression_tuned_models.items():
    pipe.fit(X_reg_train_scaled, y_reg_train)
    y_pred = pipe.predict(X_reg_test)
    results[name] = {
        "MSE": mean_squared_error(y_reg_test, y_pred),
        "R² Score": r2_score(y_reg_test, y_pred)
    }

regression_final_results = pd.DataFrame(results).T
regression_final_results.sort_values(by="R² Score", axis=0, ascending=False, inplace=True)

print("Regression Final Model Performance:")
print(regression_final_results)
print(f"\nBest Model: {regression_final_results.index[0]} with R² Score = {regression_final_results.iloc[0]['R² Score']:.4f}")

# END QUESTION

In [ ]:
# BEGIN TESTS
assert regression_final_results is not None
assert isinstance(regression_final_results, pd.DataFrame)
assert regression_final_results.shape[0] == 5
assert regression_final_results.shape[1] == 2
assert "MSE" in regression_final_results.columns
assert "R² Score" in regression_final_results.columns
# END TESTS

In [ ]:
# BEGIN QUESTION
# name: regression_comparison_plot
# points: 0.5

# Create a comparison plot for regression models
# Store figure in: regression_comparison_plot

regression_comparison_plot = plt.figure(figsize=(12, 5))

# Subplot 1: MSE Comparison
plt.subplot(1, 2, 1)
regression_final_results['MSE'].plot(kind='bar', color='steelblue')
plt.title('Mean Squared Error Comparison', fontsize=12)
plt.ylabel('MSE', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.3, axis='y')

# Subplot 2: R² Score Comparison
plt.subplot(1, 2, 2)
regression_final_results['R² Score'].plot(kind='bar', color='coral')
plt.title('R² Score Comparison', fontsize=12)
plt.ylabel('R² Score', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.3, axis='y')

plt.tight_layout()

# END QUESTION

In [ ]:
# BEGIN TESTS
assert regression_comparison_plot is not None
# END TESTS

---

## Summary

You have completed the assignment covering:

### Part A: Classification (5 marks)
- Trained 5 classification models on unbalanced data
- Tuned hyperparameters for all 5 models
- Evaluated and compared using Accuracy and ROC-AUC

### Part B: Regression (5 marks)
- Trained 5 regression models
- Tuned hyperparameters for all 5 models
- Evaluated and compared using MSE and R² Score

**Total Marks: 10 / 10**